# QanoonArena Step 3: QLoRA fine-tuning (A1) and DPO (A2)

Run this on **Google Colab with a T4 GPU** (Runtime > Change runtime type > T4 GPU).

- **A1 groups:** run sections 1 to 4, then 6.
- **A2 groups:** run sections 1 to 6 (section 5 adds DPO on top of the SFT adapter).

All training settings come from `harness/config.yaml`, so every group trains with the same
budget. Do not change them; if you must, log it in `DEVIATIONS.md`.

## 1. Settings and setup

In [ ]:
# ---- change these three lines -------------------------------------------
REPO_URL   = "https://github.com/<your-account>/<your-qanoonarena-repo>.git"
GROUP      = "G00"
TRAIN_TYPE = "A1"          # "A1" (SFT) or "A2" (SFT then DPO)
# --------------------------------------------------------------------------
import os, subprocess
if not os.path.exists("qanoonarena"):
    subprocess.run(["git", "clone", REPO_URL, "qanoonarena"], check=True)
os.chdir("qanoonarena")
print("working in", os.getcwd())

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
import sys, json, yaml, inspect, random
sys.path.insert(0, ".")
cfg = yaml.safe_load(open("harness/config.yaml"))
T = cfg["training"]
BASE = cfg["model"]["base_model"]
SEED = cfg["seeds"][0]
print("base model:", BASE)
print("training budget:", T)
import torch
assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Build training data in the chatbot's exact prompt format, and check for leakage

In [ ]:
!python -m harness.build_sft_data --out data_sft.jsonl
!python -m harness.leakage_check data_sft.jsonl

In [ ]:
from datasets import Dataset
rows = [json.loads(l) for l in open("data_sft.jsonl")]
# prompt-completion format: the loss is computed only on the answer, not on the long prompt
sft_ds = Dataset.from_list([{"prompt": r["messages"][:-1], "completion": [r["messages"][-1]]} for r in rows])
sft_ds = sft_ds.shuffle(seed=SEED)
print(sft_ds)
print(rows[0]["messages"][-1]["content"])

## 3. Load the base model in 4-bit and set up LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

def load_base():
    m = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto",
                                             torch_dtype=torch.float16)
    m.config.use_cache = False
    return m

lora = LoraConfig(r=T["lora_r"], lora_alpha=T["lora_alpha"], lora_dropout=T["lora_dropout"],
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                  task_type="CAUSAL_LM")

def make_args(cls, **kw):
    """Pass only the settings this TRL version knows (names changed between versions)."""
    names = set(getattr(cls, "__dataclass_fields__", {})) or set(inspect.signature(cls).parameters)
    if "max_length" not in names and "max_seq_length" in names and "max_length" in kw:
        kw["max_seq_length"] = kw.pop("max_length")
    dropped = [k for k in kw if k not in names]
    if dropped:
        print("note: this TRL version ignores", dropped)
    return cls(**{k: v for k, v in kw.items() if k in names})

def make_trainer(cls, **kw):
    params = inspect.signature(cls.__init__).parameters
    if "processing_class" not in params and "processing_class" in kw:
        kw["tokenizer"] = kw.pop("processing_class")
    return cls(**kw)

## 4. Supervised fine-tuning (A1, and the first stage of A2)

In [ ]:
from trl import SFTConfig, SFTTrainer

model = load_base()
sft_args = make_args(SFTConfig,
    output_dir="outputs/sft", num_train_epochs=T["epochs"], learning_rate=T["learning_rate"],
    per_device_train_batch_size=1, gradient_accumulation_steps=8, lr_scheduler_type="cosine",
    warmup_ratio=0.03, logging_steps=10, save_strategy="no", fp16=True, bf16=False,
    gradient_checkpointing=True, max_length=T["max_seq_len"], completion_only_loss=True,
    seed=SEED, report_to="none")
trainer = make_trainer(SFTTrainer, model=model, args=sft_args, train_dataset=sft_ds,
                       processing_class=tok, peft_config=lora)
import time; t0 = time.time()
trainer.train()
sft_minutes = (time.time() - t0) / 60
trainer.model.save_pretrained("adapters/a1")
tok.save_pretrained("adapters/a1")
print(f"SFT done in {sft_minutes:.1f} min -> adapters/a1")

In [ ]:
import matplotlib.pyplot as plt
hist = [h for h in trainer.state.log_history if "loss" in h]
plt.plot([h["step"] for h in hist], [h["loss"] for h in hist])
plt.xlabel("step"); plt.ylabel("training loss"); plt.title("SFT training loss")
plt.savefig("sft_loss.png", dpi=150, bbox_inches="tight"); plt.show()
print("Put sft_loss.png and the training time in MODEL_CARD.md")

## 5. DPO on top of SFT (A2 only)

Each pair has the same prompt, a **chosen** answer (the correct one from the training split)
and a **rejected** answer. The starter rule below makes rejected answers by citing a wrong
section, or by answering confidently when the right reply is "I am not sure".

**This rule is only a starting point.** Part of your Step 3 mark is the quality of your
pairs: improve the rule (for example use your SFT model's own wrong answers, or harder
near-miss sections) and document exactly how you made them in `MODEL_CARD.md`.

In [ ]:
if TRAIN_TYPE == "A2":
    from harness.common import Corpus
    corpus_ids = sorted(Corpus().docs)
    rng = random.Random(SEED)
    seeds = [json.loads(l) for l in open("data_sft.dpo_seed.jsonl")]
    pairs = []
    for s in seeds:
        chosen = json.loads(s["chosen"][0]["content"])
        if s["abstain"]:
            wrong = rng.choice(corpus_ids)
            rejected = {"answer": f"This is covered by {wrong}.", "citations": [wrong], "abstain": False}
        else:
            gold = set(s["gold_citations"])
            wrong = rng.choice([d for d in corpus_ids if d.split("@")[0] not in {g.split("@")[0] for g in gold}])
            rejected = dict(chosen, citations=[wrong])
        pairs.append({"prompt": s["prompt"],
                      "chosen": s["chosen"],
                      "rejected": [{"role": "assistant", "content": json.dumps(rejected, ensure_ascii=False)}]})
    with open("data_dpo.jsonl", "w") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(len(pairs), "preference pairs")

In [ ]:
if TRAIN_TYPE == "A2":
    !python -m harness.leakage_check data_dpo.jsonl

In [ ]:
if TRAIN_TYPE == "A2":
    from peft import PeftModel
    from trl import DPOConfig, DPOTrainer
    del model, trainer; torch.cuda.empty_cache()
    base = load_base()
    model = PeftModel.from_pretrained(base, "adapters/a1", is_trainable=True)
    dpo_ds = Dataset.from_list(pairs).shuffle(seed=SEED)
    dpo_args = make_args(DPOConfig,
        output_dir="outputs/dpo", num_train_epochs=1, learning_rate=T["dpo_learning_rate"], beta=T["dpo_beta"],
        per_device_train_batch_size=1, gradient_accumulation_steps=8, logging_steps=10, save_strategy="no",
        fp16=True, bf16=False, gradient_checkpointing=True, max_length=T["max_seq_len"],
        max_prompt_length=T["max_seq_len"] - 256, seed=SEED, report_to="none")
    # ref_model=None: TRL uses the model with the adapter switched off as the reference.
    dpo = make_trainer(DPOTrainer, model=model, ref_model=None, args=dpo_args, train_dataset=dpo_ds,
                       processing_class=tok)
    t0 = time.time(); dpo.train(); dpo_minutes = (time.time() - t0) / 60
    dpo.model.save_pretrained("adapters/a2"); tok.save_pretrained("adapters/a2")
    hist = [h for h in dpo.state.log_history if "rewards/margins" in h]
    if hist:
        plt.plot([h["step"] for h in hist], [h["rewards/margins"] for h in hist])
        plt.xlabel("step"); plt.ylabel("reward margin"); plt.title("DPO reward margin")
        plt.savefig("dpo_margin.png", dpi=150, bbox_inches="tight"); plt.show()
    print(f"DPO done in {dpo_minutes:.1f} min -> adapters/a2")

## 6. Check the adapter in the chatbot, then save it

In [ ]:
ADAPTER = "adapters/a2" if TRAIN_TYPE == "A2" else "adapters/a1"
del_names = [n for n in ("model", "base", "trainer", "dpo") if n in globals()]
for n in del_names: del globals()[n]
torch.cuda.empty_cache()
!python -m harness.evaluate --group {GROUP} --memory M0 --adapter {TRAIN_TYPE} --adapter-path {ADAPTER} --split dev --limit 10

In [ ]:
# Save the adapter: either to Google Drive ...
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/qanoonarena && cp -r {ADAPTER} /content/drive/MyDrive/qanoonarena/{GROUP}_{TRAIN_TYPE}
# ... or to Hugging Face (uncomment, and create a write token at huggingface.co/settings/tokens):
# from huggingface_hub import login; login()
# from peft import PeftModel
# PeftModel.from_pretrained(load_base(), ADAPTER).push_to_hub(f"<your-hf-user>/qanoonarena-{GROUP}-{TRAIN_TYPE}", private=True)
print("Put the link in MODEL_CARD.md")

### Running your full experiments on Colab

The same session can run Step 5. Upload your memory code first (push to GitHub, then
`!git pull`), and run:

```
!bash scripts/run_all.sh G07 M2 A1 adapters/a1
```

Download the `results/` and `traces/` folders when it finishes (or copy them to Drive).